# Clasificación del riesgo ST-KDE: Bajo, Medio y Alto / ST-KDE Risk Classification: Low, Medium, and High

**[EN]**
This document outlines how to assign the risk labels **Low**, **Medium**, and **High** to the ST-KDE probability densities.

The model does not classify automatically. It produces a density $f(s^*, t^*)$ at a point and time specified by the user. The risk levels are obtained by comparing this density with two pre-computed thresholds on the Mexico City grid.

## General Idea

1. Use the LOO-CV process configuration: Gaussian kernel, $h_s=3$ km, $h_t=8760$ h.

2. Evaluate ST-KDE only once at the **6090 centroids** of the 500 m grid.

3. Calculate the percentiles **Q33.33** and **Q66.67** of the positive densities. 4. Save these thresholds in `app/stkde_config.json`.

5. In each query, calculate only $f$ at the user's point and classify it using these fixed thresholds.

**[ES]**

Este cuaderno documenta cómo se asignan las etiquetas de riesgo **Bajo**, **Medio** y **Alto** a las densidades de probabilidad deL ST-KDE.

El modelo no clasifica por sí mismo. Produce una densidad $f(s^*, t^*)$ en un punto y un instante dado por la usuaria. Los niveles se obtienen al comparar esa densidad con dos umbrales precomputados sobre la grilla de Ciudad de México.

## Idea general

1. Usar la configuración del proceso LOO-CV : kernel gaussiano, $h_s=3$ km, $h_t=8760$ h.
2. Evaluar ST-KDE una sola vez en los **6 090 centroides** de la grilla de 500 m.
3. Calcular los percentiles **Q33.33** y **Q66.67** de las densidades positivas.
4. Guardar esos umbrales en `app/stkde_config.json`.
5. En cada consulta, calcular únicamente $f$ en el punto de la usuaria y clasificarlo con esos umbrales fijos.


## 0. Dependencias y rutas


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "app"))

from cdmx_geo import get_grid_cells
from stkde_model import (
    COL_DATE,
    COL_HOUR,
    COL_LAT,
    COL_LON,
    CONFIG_PATH,
    classify_density_score,
    get_stkde_config,
    load_stkde_config,
    _stkde_weights,
    _threshold_reference_datetime,
)

DATA_PATH = PROJECT_ROOT / "data" / "stkde_incidents.parquet"

print(f"Dataset: {DATA_PATH}")
print(f"Config:  {CONFIG_PATH}")


## 1. Cargar dataset y configuración vigente

La configuración ya contiene el kernel y los bandwidths seleccionados por LOO-CV. Si existen umbrales, se cargan; si no, el código de producción los calcularía una sola vez.


In [ ]:
df = pd.read_parquet(DATA_PATH)
df[COL_DATE] = df[COL_DATE].astype(str)
df[COL_HOUR] = df[COL_HOUR].astype(int)

config = get_stkde_config(df)

print(f"Incidentes: {len(df):,}")
print(f"Rango fechas: {df[COL_DATE].min()} — {df[COL_DATE].max()}")
print(f"Kernel: {config.kernel_name}")
print(f"h_s: {config.h_spatial_km} km")
print(f"h_t: {config.h_temporal_hours:.0f} h (≈ {config.h_temporal_hours/8760:.2f} años)")
print(f"Q33 vigente: {config.risk_q33}")
print(f"Q66 vigente: {config.risk_q66}")
print(f"Celdas de calibración: {config.threshold_cell_count}")
print(f"Referencia temporal: {config.threshold_reference_datetime}")
print(f"Método: {config.classification_method}")


## 2. Grilla CDMX de 500 m / 500 m CDMX Grid

**[EN]**

The application divides Mexico City into 500 m × 500 m cells and retains those that fall within the CDMX polygon. Result: **6090 centroids**.

Each centroid serves as a spatial reference point for estimating ST-KDE density under the same productive model configuration.

**[ES]**

La aplicación divide la Ciudad de México en celdas de 500 m × 500 m y conserva las que caen dentro del poligono de la CDMX. Resultado: **6 090 centroides**.

Cada centroide funciona como un punto de referencia espacial para estimar la densidad ST-KDE bajo la misma configuración del modelo productivo.


In [ ]:
cells = get_grid_cells()
lats = np.array([c["centroid_lat"] for c in cells], dtype=float)
lons = np.array([c["centroid_lon"] for c in cells], dtype=float)

print(f"Número de celdas: {len(cells):,}")
print(f"Latitud: [{lats.min():.4f}, {lats.max():.4f}]")
print(f"Longitud: [{lons.min():.4f}, {lons.max():.4f}]")
print(f"Costo aproximado offline: {len(cells):,} celdas × {len(df):,} incidentes ≈ {len(cells)*len(df)/1e6:.1f} M evaluaciones")


## 3. Fecha de calibración temporal

**[EN]**

Since ST-KDE is spatiotemporal, the density of each cell also depends on a reference time $t_{ref}$.

In production, the **last date in the dataset at 12:00** is used. With data from 2020–2025, this corresponds to `2025-12-31 12:00`. The idea is to calibrate the hazard surface using the most recent available time frame.

**[ES]**

Como el ST-KDE es espacio-temporal, la densidad de cada celda depende también de un instante de referencia $t_{ref}$.

En producción se usa la **última fecha del dataset a las 12:00**. Con datos 2020–2025, eso corresponde a `2025-12-31 12:00`. La idea es calibrar la superficie de riesgo en el régimen temporal más reciente disponible.


In [ ]:
ref_dt = _threshold_reference_datetime(df)
print(f"t_ref = {ref_dt.strftime('%Y-%m-%d %H:00')}")


## 4. Evaluación ST-KDE sobre las 6 090 celdas / ST-KDE Evaluation on the 6090 Cells

**[EN]**

For each centroid $s_c$, the following is calculated:

$$
f(s_c, t_{ref}) = \frac{1}{n h_s h_t} \sum_{i=1}^{n} K\left(\frac{d(s_c,s_i)}{h_s}\right) K\left(\frac{|t_{ref}-t_i|}{h_t}\right)
$$

with the kernel and bandwidths already selected. This evaluation is performed **offline only once**; it is not repeated for each user query.

**[ES]**

Para cada centroide $s_c$ se calcula:

$$
f(s_c, t_{ref}) = \frac{1}{n h_s h_t} \sum_{i=1}^{n} K\left(\frac{d(s_c,s_i)}{h_s}\right) K\left(\frac{|t_{ref}-t_i|}{h_t}\right)
$$

con el kernel y los bandwidths ya seleccionados. Esta evaluación se hace **offline una sola vez**; no se repite en cada consulta de la usuaria.


In [ ]:
# Si la config ya tiene umbrales, no es necesario volver a precomputar. / If the configuration already has thresholds, it is not necessary to recalculate.
# Esta celda reproduce el cálculo solo para documentar el procedimiento. / This cell recalculates the process only to document the procedure.
# En una computadora típica tarda del orden de 10 segundos. / On a typical computer, this takes approximately 10 seconds.

densities = _stkde_weights(lats, lons, df, ref_dt, config)
positive = densities[densities > 0]

print(f"Densidades calculadas: {len(densities):,}")
print(f"Densidades > 0: {len(positive):,}")
print(f"Mínimo: {densities.min():.6e}")
print(f"Máximo: {densities.max():.6e}")
print(f"Mediana: {np.median(densities):.6e}")


## 5. Cálculo de umbrales Q33 y Q66 / Calculation of Q33 and Q66 Thresholds 

**[EN]** 

Percentiles are calculated based on positive densities:

- **Q33.33**: separates the lower third (Low) from the middle third (Medium)
- **Q66.67**: separates the middle third (Medium) from the upper third (High)

Classification Rule:

$$
\text{risk}(f) =
\begin{cases}
\text{Low} & \text{if } f \le Q33 \\
\text{Medium} & \text{if } Q33 < f \le Q66 \\
\text{High} & \text{if } f > Q66
\end{cases}
$$

Furthermore, if $f \le 0$, **Low** is assigned.

**[ES]** 

Sobre las densidades positivas se calculan los percentiles:

- **Q33.33**: separa el tercio inferior (Bajo) del tercio intermedio (Medio)
- **Q66.67**: separa el tercio intermedio (Medio) del tercio superior (Alto)

Regla de clasificación:

$$
\text{riesgo}(f) =
\begin{cases}
\text{Bajo} & \text{si } f \le Q33 \\
\text{Medio} & \text{si } Q33 < f \le Q66 \\
\text{Alto} & \text{si } f > Q66
\end{cases}
$$

Además, si $f \le 0$, se asigna **Bajo**.


In [ ]:
q33 = float(np.percentile(positive, 33.33))
q66 = float(np.percentile(positive, 66.67))

print(f"Q33 = {q33:.8e}")
print(f"Q66 = {q66:.8e}")
print()
print("Comparación con stkde_config.json:")
print(f"  Q33 config = {config.risk_q33:.8e}")
print(f"  Q66 config = {config.risk_q66:.8e}")
print(f"  Diferencia Q33 = {abs(q33 - config.risk_q33):.3e}")
print(f"  Diferencia Q66 = {abs(q66 - config.risk_q66):.3e}")


## 6. Distribución de celdas por nivel / Cell distribution by level

**[EN]** 

Once the thresholds are calculated, the grid partitioning can be seen. Ideally, each level contains approximately one-third of the positive densities.

**[ES]** 

Una vez calculados los umbrales, se puede ver cómo queda particionada la grilla. Idealmente, cada nivel concentra aproximadamente un tercio de las densidades positivas.


In [ ]:
levels = np.array([classify_density_score(float(d), config) for d in densities])
counts = pd.Series(levels).value_counts().reindex(["Low", "Medium", "High"])
pct = (100 * counts / counts.sum()).round(1)

summary = pd.DataFrame({
    "celdas": counts,
    "porcentaje": pct,
})
summary.index = ["Bajo", "Medio", "Alto"]
summary


## 7. Cómo se usa en una consulta real / How it's used in a real-world query

**[EN]**

When the user estimates the risk of an address:

1. Google Geocoding converts the address to $(lat, lon)$.

2. ST-KDE calculates $f(s^*, t^*)$ using **all** of the 41,935 incidents.

3. `classify_density_score(f, config)` is applied with the Q33/Q66 thresholds already saved.

4. The card displays Low/Medium/High based on this comparison.

The 6,090 cells are not re-evaluated. 

**[ES]**

Cuando la usuaria estima el riesgo de una dirección:

1. Google Geocoding convierte la dirección en $(lat, lon)$.
2. ST-KDE calcula $f(s^*, t^*)$ usando **todos** los 41 935 incidentes.
3. Se aplica `classify_density_score(f, config)` con los umbrales Q33/Q66 ya guardados.
4. La tarjeta muestra Bajo / Medio / Alto según esa comparación.

No se vuelven a evaluar las 6 090 celdas.


In [ ]:
# Ejemplo ilustrativo: clasificar tres densidades hipotéticas / Illustrative example: classifying three hypothetical densities
examples = [
    1e-9,
    float(config.risk_q33),
    float((config.risk_q33 + config.risk_q66) / 2),
    float(config.risk_q66),
    float(config.risk_q66) * 2,
]

rows = []
for f in examples:
    rows.append({
        "density_score": f"{f:.3e}",
        "nivel": classify_density_score(f, config),
        "criterio": (
            "f <= 0 o f <= Q33" if f <= config.risk_q33 else
            "Q33 < f <= Q66" if f <= config.risk_q66 else
            "f > Q66"
        ),
    })

pd.DataFrame(rows)


## Resumen / Summary

**[EN]**

| Element | Value / Decision |
|---|---|
| Calibration Support | 6090 centroids of the CDMX grid |
| Reference Time | Last dataset date at 12:00 |
| Low→Medium Threshold | Q33.33 of densities > 0 |
| Medium→High Threshold | Q66.67 of densities > 0 |
| Calculation Frequency | One-time, offline |
| Use in Query | Only classify $f(s^*,t^*)$ of the point |

The current configuration is stored in `app/stkde_config.json` and is reused in `/api/estimate` without recalculating the grid.


**[ES]**

| Elemento | Valor / decisión |
|---|---|
| Soporte de calibración | 6 090 centroides de la grilla CDMX |
| Tiempo de referencia | última fecha del dataset a las 12:00 |
| Umbral Bajo→Medio | Q33.33 de densidades > 0 |
| Umbral Medio→Alto | Q66.67 de densidades > 0 |
| Frecuencia de cálculo | una sola vez, offline |
| Uso en consulta | solo clasificar $f(s^*,t^*)$ del punto |

La configuración vigente queda en `app/stkde_config.json` y se reutiliza en `/api/estimate` sin recalcular la grilla.
